Notebook for model training

In [12]:
import os
import time
import json
import catboost as cb
import pandas as pd
import numpy as np
import warnings
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')

: 

In [13]:
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
OUT_PATH   = "submission.csv"

TARGET = "urgency_level"
ID_COL = "id"
TIME_COL = "step"
CAT_COLS = ["type"]

# Load files
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# strip any accidental whitespace in column names
train.columns = train.columns.str.strip()
test.columns  = test.columns.str.strip()

# Light cleanup for columns
for c in CAT_COLS:
    if c in train.columns:
        train[c] = train[c].astype(str)
        test[c]  = test[c].astype(str)

# Drop raw IDs (high-cardinality strings)
DROP_COLS = ["nameOrig", "nameDest"]
train = train.drop(columns=[c for c in DROP_COLS if c in train.columns])
test  = test.drop(columns=[c for c in DROP_COLS if c in test.columns])

# Select features
feature_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]
X = train[feature_cols]
y = train[TARGET]
X_test = test[feature_cols]

# Time-based split (last 20% of steps as validation)
cutoff = np.quantile(train[TIME_COL], 0.80)
tr_idx = train[TIME_COL] <= cutoff
va_idx = train[TIME_COL] > cutoff

X_tr, y_tr = X.loc[tr_idx], y.loc[tr_idx]
X_va, y_va = X.loc[va_idx], y.loc[va_idx]

# ensure the categorical column is actually a string
train["type"] = train["type"].astype(str)
test["type"]  = test["type"].astype(str)

# CatBoost is most reliable with categorical feature INDICES (not names)
cat_feature_indices = [X_tr.columns.get_loc("type")]

train_pool = cb.Pool(X_tr, y_tr, cat_features=cat_feature_indices)
valid_pool = cb.Pool(X_va, y_va, cat_features=cat_feature_indices)
test_pool  = cb.Pool(X_test, cat_features=cat_feature_indices)


# Class weights helps with imbalance
counts = y_tr.value_counts().sort_index()
total = counts.sum()
weights = (total / (len(counts) * counts)).to_dict()
class_weights = [weights.get(i, 1.0) for i in range(4)]

# Train
model = cb.CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1:average=Macro;use_weights=False",
    iterations=8000,
    learning_rate=0.05,
    depth=9,
    l2_leaf_reg=5.0,
    random_seed=42,
    class_weights=class_weights,
    od_type="Iter",
    od_wait=200,
    verbose=200
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# Validate the macro-f1
va_pred = model.predict(X_va).astype(int).reshape(-1)
print("Validation macro-F1:", f1_score(y_va, va_pred, average="macro"))

# Predict + save submission
test_pred = model.predict(test_pool).astype(int).reshape(-1)
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred})
submission.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}")

0:	learn: 0.3025395	test: 0.3891073	best: 0.3891073 (0)	total: 3.36s	remaining: 7h 28m 31s
200:	learn: 0.4958513	test: 0.5824847	best: 0.5824847 (198)	total: 8m 16s	remaining: 5h 21m 5s
400:	learn: 0.5185497	test: 0.6286077	best: 0.6286077 (400)	total: 17m 28s	remaining: 5h 31m 8s
600:	learn: 0.5425583	test: 0.6612925	best: 0.6613454 (597)	total: 28m 10s	remaining: 5h 46m 55s
800:	learn: 0.5680588	test: 0.6928378	best: 0.6928378 (800)	total: 40m 46s	remaining: 6h 6m 29s


KeyboardInterrupt: 

In [ ]:
print("Best iteration:", model.get_best_iteration())
print("Best score:", model.get_best_score())


Best iteration: 123
Best score: {'learn': {'TotalF1': 0.9932699929630708, 'MultiClass': 0.02695042328108358}, 'validation': {'TotalF1': 0.9914357694915201, 'MultiClass': 0.03251987450459331}}
